In [0]:
import pyspark.sql.functions as F

In [0]:
# Table 1: Transaction Data (The "Large" Fact Table)
sales_data = [
    ("TXN-101", "Hundred Predator 79", 1, 4500.00),
    ("TXN-102", "JP70 Magnite String", 3, 1200.00),
    ("TXN-103", "Yonex BG65", 2, 900.00),
    ("TXN-104", "Hundred Predator 79", 1, 4500.00)
]
sales_cols = ["TransactionID", "ItemName", "Quantity", "TotalAmount"]

df_sales = spark.createDataFrame(data=sales_data, schema=sales_cols)

# Table 2: Product Catalog (The "Small" Dimension Table)
catalog_data = [
    ("Hundred Predator 79", "Racket", "Hundred"),
    ("JP70 Magnite String", "String", "Hundred"),
    ("Yonex BG65", "String", "Yonex"),
    ("Victor Thruster", "Racket", "Victor")
]
catalog_cols = ["ItemName", "Category", "Brand"]

df_catalog = spark.createDataFrame(catalog_data, catalog_cols)

Your Hands-On Mission
I want you to chain a pipeline that does the following:

- Start with df_sales.

- Perform an inner join with df_catalog using a broadcast join on the ItemName column.

- Filter the resulting DataFrame so it only shows items where the Brand is "Hundred".

- Select only the TransactionID, ItemName, and Category.

- Assign it to a variable and use display() to show it.

In [0]:
df_inn_joined = (df_sales
                 .join(F.broadcast(df_catalog), on='ItemName', how='inner')
                 .filter(F.col('Brand')=='Hundred')
                 ).select("TransactionID", "ItemName", "Category")

In [0]:
display(df_inn_joined)

TransactionID,ItemName,Category
TXN-101,Hundred Predator 79,Racket
TXN-102,JP70 Magnite String,String
TXN-104,Hundred Predator 79,Racket


Hands-On: Taming Nulls
- Write a single chained PySpark query that does the following:

- Start with df_repairs.

- Drop any row where MouseModel is null.

- Fill any nulls in SwitchBrand with the string "Stock Switch".

- Fill any nulls in PartCost with 0.0.

- Save the output to a variable and display() it.

In [0]:
# 1. Setup Data with Nulls (None in Python)
repair_data = [
    ("Logitech G304", "Omron", None),
    ("Razer Viper", None, 15.20),
    ("Custom Build", "Huano", 11.50),
    ("SteelSeries", None, None),
    (None, "Kailh", 8.00)
]
columns = ["MouseModel", "SwitchBrand", "PartCost"]

df_repairs = spark.createDataFrame(repair_data, columns)
df_repairs.show()

+-------------+-----------+--------+
|   MouseModel|SwitchBrand|PartCost|
+-------------+-----------+--------+
|Logitech G304|      Omron|    NULL|
|  Razer Viper|       NULL|    15.2|
| Custom Build|      Huano|    11.5|
|  SteelSeries|       NULL|    NULL|
|         NULL|      Kailh|     8.0|
+-------------+-----------+--------+



In [0]:
df_repair_final = df_repairs.dropna(subset=["MouseModel"]).fillna({'SwitchBrand':"Stock Switch", "PartCost":0})

In [0]:
display(df_repair_final)

MouseModel,SwitchBrand,PartCost
Logitech G304,Omron,0.0
Razer Viper,Stock Switch,15.2
Custom Build,Huano,11.5
SteelSeries,Stock Switch,0.0
